**PROGRAMA PARA UNIFICACIÓN Y LIMPIEZA DE DATOS** <br><br>

*WorkFlow:* <br>
--> archivos .CSV <br>
--> Importar datos a través de SQLDuckDB <br>
--> Unificación  <br>
--> Generación de Dataframe en Pandas <br>
--> Limpieza de datos <br>
--> Generación de archivo .CSV limpio<br>

  

In [12]:
# Librerías
import pandas as pd
import duckdb as duck
import re


# Importación de archivo en SQL DuckDB: "spotify"
con = duck.connect()

con.execute("""
CREATE TABLE spotify AS
SELECT *
FROM read_csv_auto('Spotify-2000-clean.csv')
""")

# Ver información de spotify
spotify = con.execute("""
SELECT *
FROM spotify
""").fetchdf()

spotify.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Title               1994 non-null   object
 1   Artist              1994 non-null   object
 2   Genre               1994 non-null   object
 3   Year                1994 non-null   int64 
 4   BPM                 1994 non-null   int64 
 5   Energy              1994 non-null   int64 
 6   Danceability        1994 non-null   int64 
 7   Loudness_db         1994 non-null   int64 
 8   Liveness            1994 non-null   int64 
 9   Valence             1994 non-null   int64 
 10  Duration            1994 non-null   int64 
 11  Acousticness        1994 non-null   int64 
 12  Speechiness         1994 non-null   int64 
 13  Popularity          1994 non-null   int64 
 14  Decade              1994 non-null   int64 
 15  Popularity_segment  1994 non-null   object
dtypes: int64(12), object(4)


In [13]:
# Importación de archivo en SQL DuckDB: "winners"

con.execute("""

CREATE TABLE winners AS
SELECT DISTINCT 
    nominee,
    artist,
    winner
FROM read_csv_auto('the_grammy_awards.csv')
""")


# Primeras filas
winners_preview = con.execute("""
SELECT *
FROM winners

""").fetchdf()

winners_preview.info()
winners_preview.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4447 entries, 0 to 4446
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   nominee  4446 non-null   object
 1   artist   2841 non-null   object
 2   winner   4447 non-null   bool  
dtypes: bool(1), object(2)
memory usage: 74.0+ KB


,nominee,artist,winner
0,Old Town Road,Lil Nas X Featuring Billy Ray Cyrus,True
1,"I,I",Bon Iver,True
2,7,Lil Nas X,True
3,Bad Guy,None,True
4,Bring My Flowers Now,None,True


<br>

*Usaremos dos columnas para hacer la unificación de bases de datos; Title + Artist (para Spotyfy) y  nominee + artist  (winners). <br>
Antes de hacer el JOIN, unificaremos el formato de escritura del contenido en esas columnas pasando todo a minúsculas, quitando espacios extras y usando _ en vez de espacios entre palabras para que se pueda hacer la unificación correctamente.* <br>


In [14]:
# Cambio de formato en celdas que se usarán como clave

con.execute("""

UPDATE spotify
SET
    Title = LOWER(REPLACE(TRIM(Title), ' ', '_')),
    Artist = LOWER(REPLACE(TRIM(Artist), ' ', '_')) ;

UPDATE winners
SET
    nominee = LOWER(REPLACE(TRIM(nominee), ' ', '_')),
    artist = LOWER(REPLACE(TRIM(artist), ' ', '_'))

    
""")

# Ver cambios implementados
preview2 = con.execute("""
SELECT *
FROM winners
LIMIT 3
""").fetchdf()

preview2

,nominee,artist,winner
0,old_town_road,lil_nas_x_featuring_billy_ray_cyrus,True
1,"i,i",bon_iver,True
2,7,lil_nas_x,True


In [15]:
# Unificación de tablas en una sola tabla: "spotify_joined"

con.execute ("""
CREATE TABLE spotify_joined AS
SELECT
    s.*,
    COALESCE(w.winner, FALSE) AS winner

FROM spotify s
LEFT JOIN winners w
    ON s.Title = w.nominee
    AND s.Artist = w.artist
    """)

df = con.execute ("""
    SELECT *
    FROM spotify_joined;
    """).fetchdf()

# Visualización de nueva tabla
df.info()
df.tail(3)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Title               1994 non-null   object
 1   Artist              1994 non-null   object
 2   Genre               1994 non-null   object
 3   Year                1994 non-null   int64 
 4   BPM                 1994 non-null   int64 
 5   Energy              1994 non-null   int64 
 6   Danceability        1994 non-null   int64 
 7   Loudness_db         1994 non-null   int64 
 8   Liveness            1994 non-null   int64 
 9   Valence             1994 non-null   int64 
 10  Duration            1994 non-null   int64 
 11  Acousticness        1994 non-null   int64 
 12  Speechiness         1994 non-null   int64 
 13  Popularity          1994 non-null   int64 
 14  Decade              1994 non-null   int64 
 15  Popularity_segment  1994 non-null   object
 16  winner              1994

,Title,Artist,Genre,Year,BPM,Energy,Danceability,Loudness_db,Liveness,Valence,Duration,Acousticness,Speechiness,Popularity,Decade,Popularity_segment,winner
1991,johnny_b._goode,chuck_berry,blues rock,1959,168,80,53,-9,31,97,162,74,7,74,1950,Alta,False
1992,take_five,the_dave_brubeck_quartet,bebop,1959,174,26,45,-13,7,60,324,54,4,65,1950,Media-Alta,False
1993,blueberry_hill,fats_domino,adult standards,1959,133,50,49,-10,16,83,148,74,3,56,1950,Media-Baja,False


In [21]:
# Cambiamos los headers a minúsculas 
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

# Cambiamos el contenido a snakecase format
df = df.map(
    lambda x: re.sub(r"\s+", "_", x.lower().strip())
    if isinstance(x, str) else x
)

# Visualizamos cambios 
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   title               1994 non-null   object
 1   artist              1994 non-null   object
 2   genre               1994 non-null   object
 3   year                1994 non-null   int64 
 4   bpm                 1994 non-null   int64 
 5   energy              1994 non-null   int64 
 6   danceability        1994 non-null   int64 
 7   loudness_db         1994 non-null   int64 
 8   liveness            1994 non-null   int64 
 9   valence             1994 non-null   int64 
 10  duration            1994 non-null   int64 
 11  acousticness        1994 non-null   int64 
 12  speechiness         1994 non-null   int64 
 13  popularity          1994 non-null   int64 
 14  decade              1994 non-null   int64 
 15  popularity_segment  1994 non-null   object
 16  winner              1994

,title,artist,genre,year,bpm,energy,danceability,loudness_db,liveness,valence,duration,acousticness,speechiness,popularity,decade,popularity_segment,winner
0,sunrise,norah_jones,adult_standards,2004,157,30,53,-14,11,68,201,94,3,71,2000,media-alta,True
1,the_pretender,foo_fighters,alternative_metal,2007,173,96,43,-4,3,37,269,0,4,76,2000,alta,True
2,without_me,eminem,detroit_hip_hop,2002,112,67,91,-3,24,66,290,0,7,82,2000,alta,True
3,uninvited,alanis_morissette,alternative_rock,2005,127,54,38,-5,9,19,276,2,3,57,2000,media-baja,True
4,cry_me_a_river,justin_timberlake,dance_pop,2002,74,65,62,-7,10,56,288,57,18,74,2000,alta,True


In [19]:
# Exportamos el dataset a un nuevo archivo .CSV
df.to_csv('spotify-2000-joined-clean.csv', index=False)